# LLMSTU — Early golden check (2 shards)
Validate your locked `config.yaml` (crop + prompt + schema) on ~2 shards BEFORE the
full 15–20h run. You'll crop+caption a small slice, hand-label ~200 crops, and get
real per-field accuracy. If it looks good → run the full pipeline; if not, fix the
prompt/schema and repeat cheaply.

Use your **~95GB GPU** runtime. Everything runs the model **on the GPU** (no API).

## 1. Setup (clone + installs + login)

In [ ]:
# PRIVATE repo? add GITHUB_TOKEN to Colab Secrets (key icon). Public clones w/o one.
REPO = "vaelkokach/LLMSTU-pipeline"
import os
from google.colab import userdata
def _secret(n):
    try: return userdata.get(n)
    except Exception: return None
if REPO and not os.path.isdir("/content/LLMSTU-pipeline"):
    gh = _secret("GITHUB_TOKEN"); auth = f"{gh}@" if gh else ""
    !git clone https://{auth}github.com/{REPO}.git /content/LLMSTU-pipeline
%cd /content/LLMSTU-pipeline
!pip install -q "ultralytics>=8.3.0" "transformers>=4.57.0" accelerate "bitsandbytes>=0.43.0" \
    "huggingface_hub>=0.35.0" pandas pyarrow pyyaml pillow
from huggingface_hub import login
login(_secret("HF_TOKEN"))

## 2. Crop 2 shards (with your locked config)
Downloads `shard_000`+`shard_001` (~10k frames) and crops with your chosen crop config.

In [ ]:
from pathlib import Path
from llmstu import crop as crop_mod, dataset_io as io
from llmstu.config import load
cfg = load('config.yaml')
print('crop_mode:', cfg.crop.crop_mode, '| prompt:', cfg.caption.prompt_name,
      '| schema:', cfg.caption.schema_name, '| model:', cfg.caption.model_id)

CAP = 800   # crops to caption (sample 200 from these for the golden set)
frames = io.download_frames(cfg.data.source_repo, cfg.data.source_subdir,
                            Path('work/frames_dl'), token=None,
                            shards=['shard_000','shard_001'])
crop_mod.run(io.iter_frames(frames, cfg.data.frames_glob), Path('work/crops'),
             cfg.crop, manifest_path=Path('work/crops_manifest.jsonl'))

## 2b. Remove near-duplicate crops
1-fps footage repeats a lot. Dedup BEFORE captioning so the 800 crops span the footage instead of ~80 consecutive seconds of the same students.

In [ ]:
from llmstu import dedup
dedup.dedup_manifest(Path('work/crops_manifest.jsonl'), Path('work/crops'),
                     Path('work/crops_manifest_dedup.jsonl'),
                     hamming_threshold=cfg.dedup.hamming_threshold,
                     cell_px=cfg.dedup.cell_px, hash_size=cfg.dedup.hash_size)

In [ ]:
from llmstu import caption as caption_mod
# caption the DEDUPED manifest (diverse crops), capped at CAP for a fast early check
caption_mod.run(Path('work/crops_manifest_dedup.jsonl'), Path('work/crops'),
                Path('work/pseudo_labels.jsonl'), cfg.caption, limit=CAP)

In [ ]:
# peek at a few labeled crops
import json, glob
from IPython.display import Image as IPImage, display
rows=[json.loads(l) for l in open('work/pseudo_labels.jsonl')]
print(len(rows),'labeled crops. sample:')
for r in rows[:3]:
    display(IPImage(f"work/crops/{r['crop_path'].split('/')[-1]}", width=180))
    print({k:r.get(k) for k in ['activity','gaze_direction','engagement_level','caption']})

## 3. Build a ~200-crop golden sample\nSelf-contained (`--embed`). Download the zip, unzip, open `index.html`.

In [ ]:
# small --min-per-class so a 200-crop set doesn't overshoot; hard subset included
!python scripts/03_sample_eval.py --labels work/pseudo_labels.jsonl --crops work/crops \
    --n 200 --stratify-fields engagement_level,activity,gaze_direction \
    --min-per-class 6 --hard-frac 0.2 --embed
!cd labeling && zip -qr /content/labeling_early.zip index.html data.json
from google.colab import files
files.download('/content/labeling_early.zip')

## 4. Hand-label (on your computer)
Unzip `labeling_early.zip`, open `index.html` (double-click — images are embedded).
For each crop the Qwen prediction is pre-filled: press **v** if it's already correct,
or fix the fields (changed ones highlight amber). Prioritise the fields the tuning
report flagged as shaky. Click **Export golden.json** when done.

Then upload `golden.json` back here (Files panel → upload into `/content/LLMSTU-pipeline/labeling/`),
or run the cell below to pick it from your machine.

In [ ]:
from google.colab import files
up = files.upload()   # choose your exported golden.json
import shutil, os
name = list(up)[0]
os.makedirs('labeling', exist_ok=True)
shutil.move(name, 'labeling/golden.json')
print('saved -> labeling/golden.json')

## 5. Score pseudo-labels vs golden\nPer-field accuracy + example disagreements. Only counts crops you reviewed.

In [ ]:
!python scripts/07_eval_golden.py --golden labeling/golden.json
from IPython.display import HTML
HTML(open('work/eval/golden_report.html').read())

## 6. Decide
- **Accuracy good** on the fields you care about → run the full pipeline
  (`LLMSTU_pipeline_colab.ipynb`, settings in `docs/GPU_95GB.md`). Keep this
  golden.json — you can grow it to ~500 after the full run.
- **A field is weak** → tighten it in `llmstu/prompts.py` / `llmstu/schema.py`
  (or pick a different `prompt_name`/`schema_name` in `config.yaml`), push, then
  re-run cells 2–5 on the same 2 shards. Cheap to iterate here.